### Import Dependencies

In [15]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

### Download an example reference data point from LangSmith

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [3]:
ls_client = Client()

In [4]:
dataset = ls_client.read_dataset(
    dataset_name="coordinator-agent-development"
)

In [5]:
dataset

Dataset(name='coordinator-agent-development', description='', data_type=<DataType.kv: 'kv'>, id=UUID('de1dfe4a-f6a5-4ce6-bf82-b4043fdac6b1'), created_at=datetime.datetime(2026, 8, 26, 16, 43, 45, 901652, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 8, 26, 16, 43, 45, 901652, tzinfo=TzInfo(0)), example_count=1, session_count=0, last_session_start_time=None, inputs_schema={'type': 'object', 'title': 'dataset_input_schema', 'required': [], 'properties': {}}, outputs_schema=None, transformations=None, metadata=None)

In [7]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))

[<class 'langsmith.schemas.Example'>(id=c453b3ab-7dba-4d6e-b9b5-2fc65bcc7e6f, dataset_id=de1dfe4a-f6a5-4ce6-bf82-b4043fdac6b1, link='https://smith.langchain.com/o/80f44ede-96d4-4bf0-975a-a012564f3081/datasets/de1dfe4a-f6a5-4ce6-bf82-b4043fdac6b1/e/c453b3ab-7dba-4d6e-b9b5-2fc65bcc7e6f')]

In [10]:
reference_inputs = [item.inputs["input"] for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [11]:
reference_inputs

[{'messages': [{'type': 'human',
    'content': 'Can you suggest me a tablet?',
    'additional_kwargs': {},
    'response_metadata': {}},
   {'id': 'resp_017e3759e60d6fb6006a7a5e8b0c0c8198a7a1e822902ea753',
    'type': 'ai',
    'content': [{'id': 'rs_017e3759e60d6fb6006a7a5e8bcfa08198ba67ee19ae8eaeba',
      'type': 'reasoning',
      'content': [],
      'summary': [],
      'encrypted_content': 'gAAAAABqel6MLbSnkpzP4ntqwi-AflOdbnIV-VsdsWJ2K1J-983rNkofy0IBIpRPcSDzEcvbcGp-CrdwmL7x0NUL26eCoyD1Bkan-pHzFYtyaanbWUkraP--qtppHJklxC-p0qy7mYN_EXiZXJDkzoI-HZoro5ah07u8DlCikA2jK2OIhDfsM7_AzWWpoHyD3ppPu-iBXYF09k5SBwshbOiiKwyfqt5flklqFbE5wX-00dtqMdidgPJwSNyvgpNTQCkKoXvIno-ymFRs5ur7DIaUAdM9H4E4bcho8RskGJsfRuKG7H_33C7aJlh52G95doHwO1lKH0bXTK8xUYMGIUqMbFEwr6kXB9V-XqLR25LHC4H0TjxPdzlW2gCa6jl4qQK9arNEV1FPHYLAbdMRblnSOnRRLMcguJfY6V14H04WDZg9L2e8WTfFLOh1KFzsQLfYb9LWIitKU-JgZUBjFI-q624xO29tKvEZUUBIA1yQiURpKjoWQ-YCkIz84X6NNB02lUdB44rluSDWO85n3tEVdVDdA3Xi4wwG4qbXsYsNBDzfQOVAmLfji4F_uJopdDrvJj2Xcn4e_B4awVX0M

In [12]:
reference_outputs = [item.outputs for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [13]:
reference_outputs

[{'answer': '',
  'coordinator_agent': {'plan': [], 'next_agent': '', 'final_answer': True}}]

### Coordinator Agent

In [22]:
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langsmith import traceable

import instructor

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode


from langchain_core.messages import SystemMessage, convert_to_openai_messages, HumanMessage, AIMessage
from IPython.display import Image, display

from typing import Any, Annotated, List
from pydantic import Field
from operator import add

from jinja2 import Template

from utils.tools import get_formatted_item_context, get_formatted_reviews_context, get_shopping_cart, remove_from_cart, add_to_shopping_cart, check_warehouse_availability, reserve_warehouse_items

In [23]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    user_intent: str = ""
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""
    user_id: str = ""
    cart_id: str = ""


In [24]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:

    #jinja template
    prompt_template ="""You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to create plans for solving user queries and delegate the tasks accordingly.
- You will be given a conversation history, your task is to create a plan for solving the user's query.
- After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
- If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.

## Examples

Question: "Do you have running shoes under $100?"
Next agent: product_qna_agent

Question: "Can you list the items in my cart?"
Next agent: shopping_cart_agent

Question: "Can you reserve my shopping cart?"
Next agent: warehouse_manager_agent
"""

    template = Template(prompt_template)
    prompt = template.render(
        user_id=state.user_id,
        cart_id=state.cart_id
    )

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )

    #llm_with_tools 
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan], #tools can be functions and pydantic model
        tool_choice="required",
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages # message list to inject everytime we are running llm
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""
   
    def sanitise_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [25]:
reference_inputs[0]["messages"]

[{'type': 'human',
  'content': 'Can you suggest me a tablet?',
  'additional_kwargs': {},
  'response_metadata': {}},
 {'id': 'resp_017e3759e60d6fb6006a7a5e8b0c0c8198a7a1e822902ea753',
  'type': 'ai',
  'content': [{'id': 'rs_017e3759e60d6fb6006a7a5e8bcfa08198ba67ee19ae8eaeba',
    'type': 'reasoning',
    'content': [],
    'summary': [],
    'encrypted_content': 'gAAAAABqel6MLbSnkpzP4ntqwi-AflOdbnIV-VsdsWJ2K1J-983rNkofy0IBIpRPcSDzEcvbcGp-CrdwmL7x0NUL26eCoyD1Bkan-pHzFYtyaanbWUkraP--qtppHJklxC-p0qy7mYN_EXiZXJDkzoI-HZoro5ah07u8DlCikA2jK2OIhDfsM7_AzWWpoHyD3ppPu-iBXYF09k5SBwshbOiiKwyfqt5flklqFbE5wX-00dtqMdidgPJwSNyvgpNTQCkKoXvIno-ymFRs5ur7DIaUAdM9H4E4bcho8RskGJsfRuKG7H_33C7aJlh52G95doHwO1lKH0bXTK8xUYMGIUqMbFEwr6kXB9V-XqLR25LHC4H0TjxPdzlW2gCa6jl4qQK9arNEV1FPHYLAbdMRblnSOnRRLMcguJfY6V14H04WDZg9L2e8WTfFLOh1KFzsQLfYb9LWIitKU-JgZUBjFI-q624xO29tKvEZUUBIA1yQiURpKjoWQ-YCkIz84X6NNB02lUdB44rluSDWO85n3tEVdVDdA3Xi4wwG4qbXsYsNBDzfQOVAmLfji4F_uJopdDrvJj2Xcn4e_B4awVX0MWlBTVwLVfvL84DyeD4qXiu_9lK2hXy5Rjd

In [ ]:
output = coordinator_agent(
    messages=reference_inputs[0]["messages"],
    #choose any default values, only thing matters is messages
    coordinator_agent = CoordinatorAgentProperties(
        iteration=0,
        final_answer=False,
        plan=[],
        next_agent=""
    )
)

In [27]:
def validate_coordinator_delegation(run, example):
    final_answer_match = run["coordinator_agent"]["final_answer"] == example["coordinator_agent"]["final_answer"]
    next_agent_match = run["coordinator_agent"]["next_agent"] == example["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [ ]:
validate_coordinator_delegation(output, reference_outputs[0])

### Run against langsmith

In [ ]:
def validate_coordinator_delegation(run, example):
    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_agent"]["final_answer"]
    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [ ]:
results_hybrid = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            )
        )
    ),
    data="coordinator-agent-development",
    evaluators=[
        validate_coordinator_delegation
    ],
    experiment_prefix="delegation",
    max_concurrency=10 # 10 runs in parallel
)

### Coordinator agent v2 (Change prompt)

In [ ]:
#TBD: update prompt
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:

    #jinja template
    prompt_template ="""
    
    """

    template = Template(prompt_template)
    prompt = template.render(
        user_id=state.user_id,
        cart_id=state.cart_id
    )

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )

    #llm_with_tools 
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan], #tools can be functions and pydantic model
        tool_choice="required",
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages # message list to inject everytime we are running llm
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""
   
    def sanitise_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results_hybrid = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            )
        )
    ),
    data="coordinator-agent-development",
    evaluators=[
        validate_coordinator_delegation
    ],
    experiment_prefix="delegation",
    max_concurrency=10 # 10 runs in parallel
)

### Coordinator agent v3 (Change reasoning effort)

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:

    #jinja template
    prompt_template ="""
    
    """

    template = Template(prompt_template)
    prompt = template.render(
        user_id=state.user_id,
        cart_id=state.cart_id
    )

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )

    #llm_with_tools 
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan], #tools can be functions and pydantic model
        tool_choice="required",
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages # message list to inject everytime we are running llm
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""
   
    def sanitise_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results_hybrid = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            )
        )
    ),
    data="coordinator-agent-development",
    evaluators=[
        validate_coordinator_delegation
    ],
    experiment_prefix="delegation",
    max_concurrency=10 # 10 runs in parallel
)

### Coordinator agent v4 (remove plan ): may be returning plan confuses the plan

In [29]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    #plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

In [ ]:
#TBD: update prompt
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:

    #jinja template
    prompt_template ="""
    
    """

    template = Template(prompt_template)
    prompt = template.render(
        user_id=state.user_id,
        cart_id=state.cart_id
    )

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )

    #llm_with_tools 
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan], #tools can be functions and pydantic model
        tool_choice="required",
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages # message list to inject everytime we are running llm
        ]
    )

    final_answer = False
    answer = ""
    #plan = []
    next_agent = ""
   
    def sanitise_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            #plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            #"plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results_hybrid = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            )
        )
    ),
    data="coordinator-agent-development",
    evaluators=[
        validate_coordinator_delegation
    ],
    experiment_prefix="delegation",
    max_concurrency=10, # 10 runs in parallel
    num_repetitions=3 #run 3 times and average out
)